In [ ]:
#############  PHASE 2################
#######STEP 5- regrassion models #################

##Purpose
#This notebook develops ordinary least squares (OLS) regression models to quantify
#relationships between climatic variables, population growth and flood impacts in Lagos and Durban.
##Objective: Identify drivers of flood severity in Lagos and Durban

##Research Context
#Regression analysis provides a quantitative framework for assessing the relative 
#contribution of precipitation, temperature and population growth to flood damages and the number of people affected.

##Input Data
#Climate variables
#Population indicators
#Flood damages
#People affected

##Methods
#Log transformation
#OLS regression
#Model diagnostics
#Residual analysis
#Goodness-of-fit evaluation

## Outputs
#Regression coefficients
#Model summaries
#Diagnostic plots
#Predicted relationships


#STEP 5a: Build regression datasets
import numpy as np
import pandas as pd


######Loading Data
rainfall = pd.read_csv("outputs/rainfall_clean.csv")
temperature = pd.read_csv("outputs/temperature_clean.csv")
pop_long = pd.read_csv("outputs/urban_population_clean.csv")
emdat = pd.read_csv("outputs/emdat_floods_clean.csv")



In [ ]:
print(type(rainfall))
print(type(temperature))
print(type(emdat))
print(type(pop_long))

In [ ]:
rainfall.head
temperature.head 
emdat.head
pop_long.head

In [ ]:
###Climate data 
#We need annual max rainfall and mean temperature per year
########Lagos Climate
lagos_climate = (
    rainfall[rainfall["City"] == "Lagos"]
    [["Year", "Precipitation_mm"]]
    .merge(
        temperature[temperature["City"] == "Lagos"]
        [["Year", "Temperature_C"]],
        on="Year"
    )
)

##########Durban Climate
durban_climate = (
    rainfall[rainfall["City"] == "Durban"]
    [["Year", "Precipitation_mm"]]
    .merge(
        temperature[temperature["City"] == "Durban"]
        [["Year", "Temperature_C"]],
        on="Year"
    )
)


In [ ]:
lagos_climate.head
durban_climate.head

In [ ]:
print(emdat.columns.tolist())
print("Damage_million_usd" in emdat.columns)
#Create damage column again
emdat["Damage_million_usd"] = (
    emdat["Total_damage_000usd"].fillna(0) / 1000
)
emdat[["Total_damage_000usd", "Damage_million_usd"]].head()


In [ ]:
print(emdat.columns.tolist())
print("Damage_million_usd" in emdat.columns)
for col in emdat.columns:
    print(repr(col))

In [ ]:
#Create damage variable if missing
if "Damage_million_usd" not in emdat.columns:
    emdat["Damage_million_usd"] = (
        pd.to_numeric(emdat["Total_damage_000usd"], errors="coerce")
        .fillna(0)
        / 1000
    )
######Lagos flood data
lagos_flood = (
    emdat[emdat["Country"] == "Nigeria"]
    .groupby("Year", as_index=False)
    .agg(
        deaths=("Deaths", "sum"),
        damage=("Damage_million_usd", "sum"),
        events=("Year", "count"),
        affected=("Total_affected", "sum")
    )
)

#######Durban flood data
durban_flood = (
    emdat[emdat["Country"] == "South Africa"]
    .groupby("Year", as_index=False)
    .agg(
        deaths=("Deaths", "sum"),
        damage=("Damage_million_usd", "sum"),
        events=("Year", "count"),
        affected=("Total_affected", "sum")
    )
)

print(lagos_flood.head())
print(durban_flood.head())

In [ ]:
######Urban population data

lagos_pop = (
    pop_long[pop_long["City"] == "Lagos"]
    [["Year", "Urban_population"]]
)

durban_pop = (
    pop_long[pop_long["City"] == "Durban"]
    [["Year", "Urban_population"]]
)
print(lagos_pop)
print(durban_pop)

In [ ]:
###Merge all data into modelling datasets
#Like R's left_join(by = "year")
####Merge Lagos

lagos_model = (
    lagos_flood
    .merge(lagos_climate, on="Year", how="left")
    .merge(lagos_pop, on="Year", how="left")
)

#####Merge Durban

durban_model = (
    durban_flood
    .merge(durban_climate, on="Year", how="left")
    .merge(durban_pop, on="Year", how="left")
)
print(lagos_model.head)
print(durban_model.head)

In [ ]:
#STEP 5b- Log Transform Variables
#Log-transform skewed outcome variables (common in flood damage modelling)
for df in [lagos_model, durban_model]:

    df["log_deaths"] = np.log1p(df["deaths"])
    
    df["log_affected"] = np.log1p(df["affected"])

    df["log_damage"] = np.log1p(df["damage"])

    df["log_population"] = np.log1p(
        df["Urban_population"]
    )
######Check Results
print("Lagos")
print(lagos_model.head())

print("\nDurban")
print(durban_model.head())

In [ ]:
print(lagos_model.columns)
print(durban_model.columns)

In [ ]:
#### STEP 5c- Run OLS regression
#Dependent variable: log(damage)
#Predictors: max_rainfall, mean_tempEATURE, urban_pop (log), event count
import statsmodels.formula.api as smf

In [ ]:
####Lagos damage model
lagos_damage_model = smf.ols(
    formula="""
    log_damage ~
    Precipitation_mm +
    Temperature_C +
    log_population
    """,
    data=lagos_model
).fit()

print(lagos_damage_model.summary())

In [ ]:
#####Durban damage model
durban_damage_model = smf.ols(
    formula="""
    log_damage ~
    Precipitation_mm +
    Temperature_C +
    log_population
    """,
    data=durban_model
).fit()

print(durban_damage_model.summary())

In [ ]:
print(lagos_model.columns.tolist())

In [ ]:
######STEP 5d-Flood-Affected population models
#These are often more stable than economic damage models.

#Lagos
lagos_affected_model = smf.ols(
    formula="""
    log_affected ~
    Precipitation_mm +
    Temperature_C +
    log_population
    """,
    data=lagos_model
).fit()

print(lagos_affected_model.summary())

In [ ]:
#Durban
durban_affected_model = smf.ols(
    formula="""
    log_affected ~
    Precipitation_mm +
    Temperature_C +
    log_population
    """,
    data=durban_model
).fit()

print(durban_affected_model.summary())

In [ ]:
############STEP 5e- Model summary table
#Clean comparison table for the report
summary = pd.DataFrame({

    "Model": [
        "Lagos Damage",
        "Durban Damage",
        "Lagos Affected",
        "Durban Affected"
    ],

    "R_squared": [

        lagos_damage_model.rsquared,
        durban_damage_model.rsquared,
        lagos_affected_model.rsquared,
        durban_affected_model.rsquared
    ],

    "Adj_R_squared": [

        lagos_damage_model.rsquared_adj,
        durban_damage_model.rsquared_adj,
        lagos_affected_model.rsquared_adj,
        durban_affected_model.rsquared_adj
    ],

    "Observations": [

        int(lagos_damage_model.nobs),
        int(durban_damage_model.nobs),
        int(lagos_affected_model.nobs),
        int(durban_affected_model.nobs)
    ]
})

summary = summary.round(3)

print("\n=== REGRESSION MODEL SUMMARY ===")
print(summary)

In [ ]:
#########STEP 5f- Coefficient comparison plot
#Visualising which predictors matter most in each city

import matplotlib.pyplot as plt

coef_df = pd.DataFrame({

    "Variable": lagos_damage_model.params.index,

    "Lagos_Damage":
        lagos_damage_model.params.values,

    "Durban_Damage":
        durban_damage_model.params.values
})

coef_df = coef_df[
    coef_df["Variable"] != "Intercept"
]

coef_df.set_index("Variable").plot(
    kind="bar",
    figsize=(10,6)
)

plt.title(
    "Flood Damage Drivers: Lagos vs Durban"
)

plt.ylabel("Coefficient")

plt.grid(True, alpha=0.3)

plt.tight_layout()

plt.savefig('step5_regression_coefficients.png', dpi=150, bbox_inches='tight')

plt.show()

In [ ]:
#For step 7 use
import joblib
joblib.dump(lagos_damage_model, "lagos_damage_model.pkl")
joblib.dump(durban_damage_model, "durban_damage_model.pkl")

In [ ]:
joblib.dump(lagos_model, "lagos_model.pkl")
joblib.dump(durban_model, "durban_model.pkl")

In [ ]:
#######Summary

##Main Outcomes
#Developed regression models for both study areas.
#Identified statistically significant predictors of flood impacts.
#Compared climate-driven and urbanisation-driven flood responses.
#Evaluated model performance and assumptions.

##Limitations
#Limited sample size reduced statistical power.
#National-scale explanatory variables may not fully represent local urban processes.

##Next Notebook
#->**06_spatial_flood_risk_mapping.ipynb**
#The next notebook incorporates topography and built surface information to map spatial flood susceptibility across Lagos and Durban.